# Session 2 — FastAPI & HTTP
## Empire & Ink: AI Mughal Art Studio

**What you'll build:** A working FastAPI server with live endpoints testable in Swagger UI — your first real web API.

---

## Setup

In [ ]:
!pip install fastapi httpx

## Theory 1 — HTTP Methods & What an API Is

The web works by sending **requests** and receiving **responses**. An **API** is a server that accepts requests and returns structured data.

| Method | Meaning | Example |
|--------|---------|--------|
| `GET` | Read data | Fetch all gallery items |
| `POST` | Create data | Upload a new painting |
| `DELETE` | Remove data | Delete a gallery item |

Status codes: `200 OK`, `201 Created`, `404 Not Found`, `422 Unprocessable Entity`

In [ ]:
request = {
    "method": "POST",
    "path": "/api/generate",
    "body": {"prompt": "A tiger in a Mughal garden"},
}

def handle_generate(body):
    return {"status": "ok", "prompt_received": body["prompt"]}

response = handle_generate(request["body"])
print("Response:", response)

## Theory 2 — JSON & Pydantic Models

**JSON** is the universal language of APIs — Python dicts serialised as text. **Pydantic** lets you define schemas so FastAPI validates inputs automatically.

In [ ]:
from pydantic import BaseModel
from typing import Optional
import json

class GenerateRequest(BaseModel):
    prompt: str
    style: Optional[str] = "akbari"
    width: Optional[int] = 1024
    height: Optional[int] = 1024

req = GenerateRequest(prompt="A falcon perched on a prince's wrist")
print(req)
print(json.dumps(req.model_dump(), indent=2))

## Theory 3 — FastAPI Decorators & Swagger UI

`@app.get("/path")` tells FastAPI which function handles which URL. FastAPI auto-generates interactive docs at `/docs` — this is **Swagger UI**.

In [ ]:
from fastapi import FastAPI

app = FastAPI(title="Empire & Ink API", version="1.0.0")

@app.get("/api/health")
def health_check():
    return {"status": "healthy", "app": "Empire & Ink"}

@app.get("/api/gallery")
def get_gallery():
    return {"items": [], "total": 0}

print("Routes:", [r.path for r in app.routes])

---
## In-Class Exercises

### Exercise 1 — GET /api/health endpoint

In [ ]:
from fastapi import FastAPI
from fastapi.testclient import TestClient

app = FastAPI()

# YOUR CODE HERE — add GET /api/health returning {"status": "healthy", "version": "1.0.0"}
@app.get("/api/health")
def health():
    pass

client = TestClient(app)
response = client.get("/api/health")
print(response.status_code)  # should be 200
print(response.json())

### Exercise 2 — GenerateRequest Pydantic model

In [ ]:
from pydantic import BaseModel
from typing import Optional

# YOUR CODE HERE — define GenerateRequest with:
# - prompt: str (required)
# - style: Optional[str] defaulting to "akbari"
# - width: Optional[int] defaulting to 1024
# - height: Optional[int] defaulting to 1024
class GenerateRequest(BaseModel):
    pass

req = GenerateRequest(prompt="Elephants crossing the Ganges")
print(req)

### Exercise 3 — POST /api/generate stub

In [ ]:
from fastapi import FastAPI
from fastapi.testclient import TestClient
from pydantic import BaseModel
from typing import Optional

app = FastAPI()

class GenerateRequest(BaseModel):
    prompt: str
    style: Optional[str] = "akbari"

# YOUR CODE HERE — POST /api/generate returns {"status":"queued","prompt":...,"style":...}
@app.post("/api/generate")
def generate(req: GenerateRequest):
    pass

client = TestClient(app)
r = client.post("/api/generate", json={"prompt": "A royal hunt scene"})
print(r.status_code)
print(r.json())

### Exercise 4 — GET /api/gallery returning fake items

In [ ]:
FAKE_GALLERY = [
    {"id": "1", "title": "Baburnama Folio", "image_url": "https://example.com/1.jpg"},
    {"id": "2", "title": "Akbarnama Scene", "image_url": "https://example.com/2.jpg"},
]

# YOUR CODE HERE — GET /api/gallery returns {"items": FAKE_GALLERY}
@app.get("/api/gallery")
def get_gallery():
    pass

r = client.get("/api/gallery")
print(r.json())

### Exercise 5 — Test all routes with TestClient

In [ ]:
routes_to_test = [
    ("GET",  "/api/health",  None),
    ("GET",  "/api/gallery", None),
    ("POST", "/api/generate", {"prompt": "Peacock in a royal garden"}),
]
for method, path, body in routes_to_test:
    resp = client.get(path) if method == "GET" else client.post(path, json=body)
    print(f"{method} {path} -> {resp.status_code}: {resp.json()}")

---
## Post-Class Exercises

### Challenge 1 — Input validation with Pydantic

In [ ]:
from pydantic import BaseModel, field_validator
from typing import Optional

VALID_STYLES = ["akbari", "jahangiri", "shahjahani"]

class GenerateRequest(BaseModel):
    prompt: str
    style: Optional[str] = "akbari"
    width: Optional[int] = 1024
    height: Optional[int] = 1024

    # YOUR CODE HERE — raise ValueError if style not in VALID_STYLES
    @field_validator("style")
    @classmethod
    def validate_style(cls, v):
        pass

try:
    bad = GenerateRequest(prompt="test", style="renaissance")
except Exception as e:
    print("Validation caught:", e)

good = GenerateRequest(prompt="test", style="akbari")
print("Valid:", good)

### Challenge 2 — DELETE /api/gallery/{id}

In [ ]:
from fastapi import FastAPI, HTTPException
from fastapi.testclient import TestClient

app = FastAPI()
fake_db = {"1": {"title": "Baburnama"}, "2": {"title": "Akbarnama"}}

# YOUR CODE HERE — DELETE /api/gallery/{item_id}
# Returns {"deleted": True, "id": item_id} or raises 404
@app.delete("/api/gallery/{item_id}")
def delete_item(item_id: str):
    pass

client = TestClient(app)
print(client.delete("/api/gallery/1").json())
print(client.delete("/api/gallery/99").json())

### Challenge 3 — GET /api/gallery/{id} with 404

In [ ]:
# YOUR CODE HERE — GET /api/gallery/{item_id}
# Returns the item dict or raises HTTPException(404)
@app.get("/api/gallery/{item_id}")
def get_item(item_id: str):
    pass

client2 = TestClient(app)
print(client2.get("/api/gallery/2").json())
print(client2.get("/api/gallery/999").json())

### Challenge 4 — Full test suite

In [ ]:
def run_test_suite(client):
    tests = []
    # YOUR CODE HERE — test all 4 scenarios, append (description, passed: bool)
    # 1. GET /api/gallery -> 200
    # 2. POST /api/generate with valid body -> 200
    # 3. POST /api/generate with invalid style -> 422
    # 4. GET /api/gallery/nonexistent -> 404
    return tests

for desc, passed in run_test_suite(client):
    print(f"[{'PASS' if passed else 'FAIL'}] {desc}")

---
## What you built today

- Understood HTTP methods: GET, POST, DELETE and status codes
- Defined Pydantic models to validate API inputs automatically
- Created FastAPI route handlers with decorators
- Tested every endpoint with TestClient — no browser needed

**Next session:** Session 3 — Supabase & Database — store your gallery permanently in a real cloud database!